<a href="https://colab.research.google.com/github/nagayada/Nag_New_Repo/blob/main/Mini_Assignment_2_Y_Nagaraju.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, StringType, DoubleType

### Initialize Spark Session

In [2]:
spark = SparkSession.builder.appName("FlightAnalysis").getOrCreate()

### Load Dataset

In [3]:
file_path = "/content/Flight Dataset - CSV(in).csv"
df = spark.read.csv(file_path, header=True, inferSchema=True)
df.printSchema()
display(df.limit(5).toPandas())

root
 |-- FL_DATE: string (nullable = true)
 |-- DEP_DELAY: integer (nullable = true)
 |-- ARR_DELAY: integer (nullable = true)
 |-- AIR_TIME: integer (nullable = true)
 |-- DISTANCE: integer (nullable = true)
 |-- DEP_TIME: double (nullable = true)
 |-- ARR_TIME: double (nullable = true)



,FL_DATE,DEP_DELAY,ARR_DELAY,AIR_TIME,DISTANCE,DEP_TIME,ARR_TIME
0,1/1/2006,5,19,350,2475,9.083333,12.483334
1,1/2/2006,167,216,343,2475,11.783334,15.766666
2,1/3/2006,-7,-2,344,2475,8.883333,12.133333
3,1/4/2006,-5,-13,331,2475,8.916667,11.950000
4,1/5/2006,-3,-17,321,2475,8.950000,11.883333


### Task 1: Create a function that gives back how many flights arrived earlier than expected.

In [6]:
def flights_arrived_earlier_than_expected(dataframe):
    # 'ArrDelay' is positive for delays, negative for early arrivals
    early_flights_count = dataframe.filter(F.col("ARR_DELAY") < 0).count()
    return early_flights_count

# Example usage for Task 1
early_arrivals = flights_arrived_earlier_than_expected(df)
print(f"Number of flights that arrived earlier than expected: {early_arrivals}")

Number of flights that arrived earlier than expected: 534655


### Task 2: Create a function that determines the typical departure time for flights over 2000 miles.

In [9]:
def typical_departure_time_for_long_flights(dataframe, distance_threshold=2000):
    long_flights = dataframe.filter(F.col("DISTANCE") > distance_threshold)


    long_flights = long_flights.withColumn(
        "DEP_TIME",
        (F.floor(F.col("DEP_TIME") / 100) * 60) + (F.col("DEP_TIME") % 100)
    )

    typical_dep_time_minutes = long_flights.agg(F.mean("DEP_TIME")).collect()[0][0]

    if typical_dep_time_minutes is None:
        return "No flights found over the specified distance."

    # Convert average minutes back to HHMM format for better readability
    avg_hours = int(typical_dep_time_minutes // 60)
    avg_minutes = int(typical_dep_time_minutes % 60)
    typical_dep_time_hhmm = f"{avg_hours:02d}{avg_minutes:02d}"

    return typical_dep_time_hhmm

# Example usage for Task 2
typical_dep_time = typical_departure_time_for_long_flights(df)
print(f"Typical departure time for flights over 2000 miles (HHMM): {typical_dep_time}")

Typical departure time for flights over 2000 miles (HHMM): 0013


### Task 3: Create a function that gives back the proportion of flights that have arrival delays longer than 60 minutes.

In [10]:
def proportion_of_flights_with_long_arrival_delays(dataframe, delay_threshold=60):
    total_flights = dataframe.count()
    if total_flights == 0:
        return 0.0

    long_delay_flights = dataframe.filter(F.col("ARR_DELAY") > delay_threshold).count()
    proportion = long_delay_flights / total_flights
    return proportion

# Example usage for Task 3
long_delay_proportion = proportion_of_flights_with_long_arrival_delays(df)
print(f"Proportion of flights with arrival delays longer than 60 minutes: {long_delay_proportion:.4f}")

Proportion of flights with arrival delays longer than 60 minutes: 0.0531


### Task 4: Create a function that gives the average airtime for flights that left earlier than 9:00 am.

In [15]:
def average_airtime_for_early_morning_flights(dataframe, dep_time_threshold=900):
    # Filter flights that left earlier than the threshold (e.g., 900 for 9:00 AM)
    early_morning_flights = dataframe.filter(F.col("DEP_TIME") < dep_time_threshold)

    # Calculate the average AirTime for these flights
    avg_airtime = early_morning_flights.agg(F.mean("AIR_TIME")).collect()[0][0]
    return avg_airtime



In [17]:
# Example usage for Task 4
churned_customers = average_airtime_for_early_morning_flights(df)
print(f"Average airtime for flights that left earlier than 9:00 am: {churned_customers:.2f} minutes")

Average airtime for flights that left earlier than 9:00 am: 105.81 minutes


### Task 5: Create a function that determines the maximum arrival delay for flights that did not experience a delay upon departure.

In [19]:
def max_arrival_delay_for_on_time_departures(dataframe):
    # Filter flights with no departure delay (DepDelay <= 0)
    on_time_departures = dataframe.filter(F.col("DEP_DELAY") <= 0)

    # Find the maximum arrival delay among these flights
    max_arr_delay = on_time_departures.agg(F.max("ARR_DELAY")).collect()[0][0]
    return max_arr_delay

# Example usage for Task 5
max_delay_on_time_dep = max_arrival_delay_for_on_time_departures(df)
print(f"Maximum arrival delay for flights with no departure delay: {max_delay_on_time_dep} minutes")

Maximum arrival delay for flights with no departure delay: 701 minutes
